In [1]:
!pip install -q --upgrade ipython

In [2]:
%cd /content/drive/MyDrive/auto-hh/src
%load_ext autoreload
%autoreload 2

/content/drive/MyDrive/auto-hh/src


In [3]:
!pip install -q faiss-cpu

In [31]:
import torch
from models import BiEncoder, CrossEncoder
from vector_store import load_vector_store, create_vector_store
from core import Retriever

In [5]:
MODEL_PATH = "/content/drive/MyDrive/auto-hh/models/BiEncoder"
DATA_PATH = "/content/drive/MyDrive/auto-hh/hh_dataset_sample.csv"
INDEX_PATH = "/content/drive/MyDrive/auto-hh/faiss_index"

In [6]:
print("Загрузка би энкодера...")

bi_encoder = BiEncoder.load_trained(
    MODEL_PATH,
    "BAAI/bge-m3",
    need_attention=True
)
bi_encoder.eval()
if torch.cuda.is_available():
    bi_encoder = bi_encoder.to('cuda')

print("✅ BiEncoder готов")

Загрузка би энкодера...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BiEncoder готов


In [32]:
print("📦 Создание индекса...")

create_vector_store(bi_encoder, DATA_PATH, INDEX_PATH)

📦 Создание индекса...
📦 Создание FAISS индекса...
✅ Текстов: 20
🔢 Векторизация...
✅ Эмбеддинги: (20, 1024)
✅ Индекс: 20 векторов
✅ Тексты сохранены: 20
✅ Сохранено в: /content/drive/MyDrive/auto-hh/faiss_index


<faiss.swigfaiss_avx512.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x787a23fc7720> >

In [8]:
print("📥 Загрузка Cross-Encoder...")
cross_encoder = CrossEncoder("BAAI/bge-reranker-v2-m3")

📥 Загрузка CrossEncoder...
🔄 Загрузка Cross-Encoder: BAAI/bge-reranker-v2-m3...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Cross-Encoder готов!
✅ CrossEncoder готов


In [38]:
from pathlib import Path

faiss_files = list(Path(INDEX_PATH).iterdir())
print(f"📁 Файлы в {INDEX_PATH}:")
for f in faiss_files:
    size_kb = f.stat().st_size / 1024
    print(f"   - {f.name} ({size_kb:.1f} KB)")

📁 Файлы в /content/drive/MyDrive/auto-hh/faiss_index:
   - vacancy_index.faiss (80.0 KB)
   - vacancy_ids.npy (0.3 KB)
   - vacancy_meta.json (2.6 KB)
   - vacancy_texts.npy (81.4 KB)


In [179]:
store = load_vector_store(INDEX_PATH, bi_encoder)

print(f"✅ vacancy_texts в store: {'vacancy_texts' in store}")
print(f"✅ Количество текстов: {len(store['vacancy_texts']) if store['vacancy_texts'] else 0}")
print(f"\n📄 Пример текста [0]:")
print(store['vacancy_texts'][0][:100] if store['vacancy_texts'] else "НЕТ ТЕКСТОВ")

📥 Загрузка FAISS индекса...
✅ Тексты загружены и очищены: 20
✅ Загружено: 20 векторов
✅ vacancy_texts в store: True
✅ Количество текстов: 20

📄 Пример текста [0]:
ВАКАНСИЯ: Официант. НАВЫКИ: . ОПИСАНИЕ: Навыки: Работа в команде, Материальная ответственность, Пова


In [141]:
type(store["vacancy_texts"][8])

str

In [87]:
retriever = Retriever(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    store=store,
    retrieval_top_k=10,
    final_top_k=3,
    min_score=0.0,
)

print(f"✅ Retriever готов")

✅ Retriever готов


In [228]:
query = "Python разработчик Django"
results = retriever.search(query)

print(f"\n🔍 Запрос: {query}\n")
for i, res in enumerate(results, 1):
    print(f"{i}. Score: {res['score']:.4f} | ID: {res['vacancy_id']}")
    print(f"   Роль: {res['target_role']} | Grade: {res['grade']}")
    print(f"   Компания: {res['company']}\n")

{'idx': 15, 'vacancy_id': 130536576, 'score': 0.60618656873703, 'text': 'ВАКАНСИЯ: PHP Developer. НАВЫКИ: . ОПИСАНИЕ: Обязанности: Создание бэкенда на PHP (Bitrix) Участие в интеграции фронтенда и бэкенда Требования: Уверенное знание PHP, опыт работы с MySQL Знание принципов ООП, опыт проектирования с использованием паттернов и соблюдение принципов SOLID Опыт работы с «1C-БУС» на уровне написания компонентов и модулей (знание Bitrix framework) Знание ядра Bitrix D7, ORM, опыт работы с новым ядром Опыт работы с Elasticsearch, Kibana Опыт работы с пакетным менеджером composer Опыт работы с распределенной системой контроля версий (GIT) Опыт использования Swagger для генерации документации Опыт работы с чужим кодом Будет плюсом: Опыт работы с ORM Doctrine или другой ORM, к примеру Eloquent Опыт работы с Redis Опыт работы c Gearman Опыт работы c RabbitMQ Опыт разработки back-end кода на основе микросервисной архитектуры Опыт написания Unit тестов Мы предлагаем: Возможность работать удаленно